# 🔍 آموزش (Fine-tuning) مدل MiDaS برای تخمین عمق

این نوت‌بوک به‌صورت end-to-end این کارها را انجام می‌دهد:
1. دانلود یک **زیرمجموعه کوچک** از دیتاست NYU Depth V2 (بدون دانلود کل ۳۵ گیگابایت)
2. پیش‌پردازش داده‌ها
3. Fine-tune کردن مدل از پیش‌آموزش‌دیده `MiDaS_small`
4. ارزیابی روی داده تست با معیارهای استاندارد (AbsRel, RMSE, δ1)
5. ذخیره مدل نهایی

## ⚠️ درباره سیستم شما
این تنظیمات مخصوص سیستم زیر انتخاب شده‌اند:

| مشخصه | مقدار |
|---|---|
| CPU | Intel i7-1065G7 (4 هسته / 8 رشته) |
| RAM | 16 GB |
| GPU | NVIDIA GeForce **MX250 (2GB VRAM)** — بسیار محدود |
| OS | Ubuntu 26.04 |

چون GPU شما فقط **۲ گیگابایت VRAM** دارد و ممکن است مشکل سازگاری CUDA داشته باشد:
- از مدل سبک `MiDaS_small` استفاده می‌شود
- رزولوشن ورودی و batch size کوچک نگه داشته شده
- بک‌بون فریز می‌شود و فقط decoder آموزش می‌بیند
- `num_workers=0` برای جلوگیری از خطای Pickling
- تشخیص امن دستگاه (fallback خودکار به CPU در صورت خطای CUDA kernel)
- هم‌اندازه کردن خودکار خروجی مدل با عمق واقعی

> 💡 اگر Out of Memory دیدید، `batch_size` را به ۱ و `img_size` را به ۲۲۴ کاهش دهید.


## ۰) تست سریع مدل پیش‌آموزش‌دیده (قبل از Fine-tuning)

In [ ]:
# نصب پیش‌نیازها (فقط بار اول لازم است)
!pip install -q torch torchvision timm opencv-python-headless datasets huggingface_hub matplotlib tqdm pillow


In [ ]:
import cv2
import torch
import urllib.request
import matplotlib.pyplot as plt

def get_safe_device():
    """تشخیص امن دستگاه — در صورت خطای CUDA kernel به CPU برمی‌گردد."""
    if not torch.cuda.is_available():
        print("CUDA در دسترس نیست → استفاده از CPU")
        return torch.device("cpu")
    try:
        x = torch.randn(1, 3, 32, 32, device="cuda")
        _ = torch.nn.functional.relu(x)
        del x
        torch.cuda.empty_cache()
        print(f"CUDA کار می‌کند → GPU: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    except Exception as e:
        print(f"CUDA خطا داد ({type(e).__name__}) → fallback به CPU")
        print("دلیل احتمالی: نسخه PyTorch با معماری MX250 سازگار نیست.")
        return torch.device("cpu")

# 1) دانلود یک تصویر نمونه
url, filename = ("https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")
urllib.request.urlretrieve(url, filename)

# 2) انتخاب مدل
model_type = "MiDaS_small"
midas_pretrained = torch.hub.load("intel-isl/MiDaS", model_type)

# 3) انتخاب دستگاه امن
device_quick = get_safe_device()
midas_pretrained.to(device_quick)
midas_pretrained.eval()

# 4) بارگذاری transform
midas_transforms_quick = torch.hub.load("intel-isl/MiDaS", "transforms")
transform_quick = midas_transforms_quick.small_transform

# 5) خواندن تصویر
img_quick = cv2.imread(filename)
img_quick = cv2.cvtColor(img_quick, cv2.COLOR_BGR2RGB)
input_batch = transform_quick(img_quick).to(device_quick)

# 6) پیش‌بینی عمق
with torch.no_grad():
    prediction = midas_pretrained(input_batch)
    prediction = torch.nn.functional.interpolate(
        prediction.unsqueeze(1),
        size=img_quick.shape[:2],
        mode="bicubic",
        align_corners=False,
    ).squeeze()

output_quick = prediction.cpu().numpy()

# 7) نمایش نتیجه
plt.imshow(output_quick, cmap="inferno")
plt.colorbar(label="Relative Depth")
plt.axis("off")
plt.savefig("depth_output_pretrained.png", bbox_inches="tight")
plt.show()

# پاک‌سازی حافظه
del midas_pretrained
if device_quick.type == "cuda":
    torch.cuda.empty_cache()


## ۱) تنظیمات (Config) — متناسب با سخت‌افزار شما

In [ ]:
import os, random, time
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CONFIG = {
    "model_type": "MiDaS_small",   # سبک‌ترین مدل MiDaS
    "img_size": 256,               # اندازه ورودی
    "batch_size": 2,               # برای MX250 / CPU. در صورت OOM به 1 کاهش دهید
    "num_train_samples": 300,
    "num_val_samples": 60,
    "num_test_samples": 40,
    "epochs": 3,
    "lr": 1e-5,
    "freeze_backbone": True,       # فقط decoder آموزش می‌بیند
    "use_amp": False,              # روی CPU یا در صورت مشکل CUDA خاموش
    "num_workers": 0,              # ★ حتماً 0 — جلوگیری از PicklingError
    "data_dir": "./midas_data",
    "checkpoint_dir": "./midas_checkpoints",
}

def get_safe_device():
    if not torch.cuda.is_available():
        print("CUDA در دسترس نیست → استفاده از CPU")
        return torch.device("cpu")
    try:
        x = torch.randn(1, 3, 32, 32, device="cuda")
        _ = torch.nn.functional.relu(x)
        del x
        torch.cuda.empty_cache()
        print(f"CUDA کار می‌کند → {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    except Exception as e:
        print(f"CUDA خطا داد ({type(e).__name__}) → fallback به CPU")
        return torch.device("cpu")

device = get_safe_device()
print("Device نهایی:", device)

# اگر روی CPU هستیم، AMP را خاموش نگه می‌داریم
if device.type != "cuda":
    CONFIG["use_amp"] = False


## ۲) دانلود زیرمجموعه کوچکی از دیتاست (Streaming + Skip if exists)

از `revision="refs/convert/parquet"` استفاده می‌شود تا با نسخه‌های جدید `datasets` کار کند.
اگر داده‌ها از قبل دانلود شده باشند، دوباره دانلود نمی‌شود.

In [ ]:
from datasets import load_dataset
from PIL import Image

os.makedirs(CONFIG["data_dir"], exist_ok=True)

def already_downloaded(out_dir, n_samples):
    img_dir = os.path.join(out_dir, "images")
    depth_dir = os.path.join(out_dir, "depth")
    if not (os.path.isdir(img_dir) and os.path.isdir(depth_dir)):
        return False
    n_img = len([f for f in os.listdir(img_dir) if f.endswith(".png")])
    n_depth = len([f for f in os.listdir(depth_dir) if f.endswith(".npy")])
    return n_img >= n_samples and n_depth >= n_samples

def save_subset(iterable_ds, n_samples, out_dir):
    if already_downloaded(out_dir, n_samples):
        print(f"⏭ داده‌ها از قبل در {out_dir} موجود است ({n_samples} نمونه) — از دانلود صرف‌نظر شد")
        return

    img_dir = os.path.join(out_dir, "images")
    depth_dir = os.path.join(out_dir, "depth")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(depth_dir, exist_ok=True)

    count = 0
    for sample in iterable_ds:
        img = sample["image"].convert("RGB")
        depth = np.array(sample["depth_map"], dtype=np.float32)

        img.save(os.path.join(img_dir, f"{count:05d}.png"))
        np.save(os.path.join(depth_dir, f"{count:05d}.npy"), depth)
        count += 1
        if count % 50 == 0:
            print(f"  {out_dir}: {count}/{n_samples}")
        if count >= n_samples:
            break
    print(f"تمام شد -> {count} نمونه در {out_dir} ذخیره شد")

print("در حال دانلود زیرمجموعه train ...")
train_dir = os.path.join(CONFIG["data_dir"], "train")
if already_downloaded(train_dir, CONFIG["num_train_samples"]):
    print(f"⏭ داده‌های train از قبل موجود است — از دانلود صرف‌نظر شد")
else:
    train_stream = load_dataset(
        "sayakpaul/nyu_depth_v2",
        revision="refs/convert/parquet",
        split="train",
        streaming=True
    )
    save_subset(train_stream.take(CONFIG["num_train_samples"]),
                CONFIG["num_train_samples"], train_dir)

print("در حال دانلود زیرمجموعه validation/test ...")
val_dir = os.path.join(CONFIG["data_dir"], "val")
test_dir = os.path.join(CONFIG["data_dir"], "test")

if already_downloaded(val_dir, CONFIG["num_val_samples"]) and \
   already_downloaded(test_dir, CONFIG["num_test_samples"]):
    print(f"⏭ داده‌های val و test از قبل موجود هستند — از دانلود صرف‌نظر شد")
else:
    val_stream = load_dataset(
        "sayakpaul/nyu_depth_v2",
        revision="refs/convert/parquet",
        split="validation",
        streaming=True
    )
    save_subset(val_stream.take(CONFIG["num_val_samples"]),
                CONFIG["num_val_samples"], val_dir)
    save_subset(val_stream.skip(CONFIG["num_val_samples"]).take(CONFIG["num_test_samples"]),
                CONFIG["num_test_samples"], test_dir)


## ۳) کلاس Dataset و پیش‌پردازش

In [ ]:
import cv2
from torch.utils.data import Dataset, DataLoader

def load_midas_transform(model_type):
    transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
    if model_type in ("DPT_Large", "DPT_Hybrid"):
        return transforms.dpt_transform
    return transforms.small_transform

class NYUDepthDataset(Dataset):
    def __init__(self, root_dir, transform, img_size):
        self.img_dir = os.path.join(root_dir, "images")
        self.depth_dir = os.path.join(root_dir, "depth")
        self.files = sorted([f for f in os.listdir(self.img_dir) if f.endswith(".png")])
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = cv2.imread(os.path.join(self.img_dir, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        depth = np.load(os.path.join(self.depth_dir, fname.replace(".png", ".npy"))).astype(np.float32)

        # عمق را هم‌اندازه ورودی مدل می‌کنیم
        depth_resized = cv2.resize(depth, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        img_t = self.transform(img).squeeze(0)
        depth_t = torch.from_numpy(depth_resized)
        mask_t = (depth_t > 0).float()

        return img_t, depth_t, mask_t


## ۴) ساخت DataLoaderها

**مهم:** `num_workers` حتماً باید ۰ باشد.

In [ ]:
transform = load_midas_transform(CONFIG["model_type"])

train_dataset = NYUDepthDataset(os.path.join(CONFIG["data_dir"], "train"), transform, CONFIG["img_size"])
val_dataset   = NYUDepthDataset(os.path.join(CONFIG["data_dir"], "val"),   transform, CONFIG["img_size"])
test_dataset  = NYUDepthDataset(os.path.join(CONFIG["data_dir"], "test"),  transform, CONFIG["img_size"])

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=(device.type == "cuda")
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"num_workers = {CONFIG['num_workers']} (باید 0 باشد)")


## ۵) نمایش چند نمونه از داده (پیش از آموزش)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i in range(min(4, len(train_dataset))):
    img_t, depth_t, mask_t = train_dataset[i]
    img_show = img_t.permute(1, 2, 0).numpy()
    img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min() + 1e-6)

    axes[0, i].imshow(img_show)
    axes[0, i].set_title("RGB")
    axes[0, i].axis("off")

    axes[1, i].imshow(depth_t.numpy(), cmap="inferno")
    axes[1, i].set_title("Depth GT")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()


## ۶) بارگذاری مدل پیش‌آموزش‌دیده MiDaS

In [ ]:
midas = torch.hub.load("intel-isl/MiDaS", CONFIG["model_type"])
midas.to(device)

if CONFIG["freeze_backbone"]:
    frozen, trainable = 0, 0
    for name, param in midas.named_parameters():
        if "pretrained" in name:   # بخش بک‌بون در MidasNet_small
            param.requires_grad = False
            frozen += param.numel()
        else:
            trainable += param.numel()
    print(f"بک‌بون فریز شد. پارامترهای فریزشده: {frozen:,} | پارامترهای قابل‌آموزش: {trainable:,}")

trainable_params = [p for p in midas.parameters() if p.requires_grad]
print(f"تعداد پارامترهای قابل آموزش: {sum(p.numel() for p in trainable_params):,}")


## ۷) تابع Loss اختصاصی MiDaS (Scale-Shift Invariant)

چون خروجی مدل و عمق واقعی ممکن است اندازه متفاوتی داشته باشند، قبل از محاسبه loss پیش‌بینی را به اندازه target interpolate می‌کنیم.

In [ ]:
def compute_scale_and_shift(prediction, target, mask):
    """
    prediction و target را هم‌اندازه می‌کند، سپس scale و shift بهینه را محاسبه می‌کند.
    برمی‌گرداند: scale, shift, prediction (هم‌اندازه شده)
    """
    # اطمینان از هم‌اندازه بودن
    if prediction.shape[-2:] != target.shape[-2:]:
        if prediction.dim() == 3:
            prediction = prediction.unsqueeze(1)
        prediction = torch.nn.functional.interpolate(
            prediction,
            size=target.shape[-2:],
            mode="bilinear",
            align_corners=False
        ).squeeze(1)

    a_00 = torch.sum(mask * prediction * prediction, (1, 2))
    a_01 = torch.sum(mask * prediction, (1, 2))
    a_11 = torch.sum(mask, (1, 2))
    b_0  = torch.sum(mask * prediction * target, (1, 2))
    b_1  = torch.sum(mask * target, (1, 2))

    det = a_00 * a_11 - a_01 * a_01
    scale = torch.zeros_like(b_0)
    shift = torch.zeros_like(b_1)
    valid = det.nonzero(as_tuple=True)
    scale[valid] = (a_11[valid] * b_0[valid] - a_01[valid] * b_1[valid]) / det[valid]
    shift[valid] = (-a_01[valid] * b_0[valid] + a_00[valid] * b_1[valid]) / det[valid]
    return scale, shift, prediction


def ssi_loss(prediction, target, mask):
    scale, shift, prediction = compute_scale_and_shift(prediction, target, mask)
    aligned = scale.view(-1, 1, 1) * prediction + shift.view(-1, 1, 1)
    diff = (aligned - target) ** 2 * mask
    return diff.sum() / (mask.sum() + 1e-6)


## ۸) حلقه آموزش (Fine-tuning)

In [ ]:
from tqdm.auto import tqdm

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
optimizer = torch.optim.AdamW(trainable_params, lr=CONFIG["lr"])
scaler = torch.cuda.amp.GradScaler(enabled=(CONFIG["use_amp"] and device.type == "cuda"))

history = {"train_loss": [], "val_loss": []}

for epoch in range(CONFIG["epochs"]):
    midas.train()
    epoch_loss, t0 = 0.0, time.time()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}")
    for img, depth_gt, mask in pbar:
        img = img.to(device)
        depth_gt = depth_gt.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(CONFIG["use_amp"] and device.type == "cuda")):
            pred = midas(img)
            loss = ssi_loss(pred, depth_gt, mask)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train = epoch_loss / len(train_loader)

    # Validation
    midas.eval()
    val_loss = 0.0
    with torch.no_grad():
        for img, depth_gt, mask in val_loader:
            img = img.to(device)
            depth_gt = depth_gt.to(device)
            mask = mask.to(device)
            pred = midas(img)
            val_loss += ssi_loss(pred, depth_gt, mask).item()
    avg_val = val_loss / max(len(val_loader), 1)

    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)
    print(f"Epoch {epoch+1}: train_loss={avg_train:.4f}  val_loss={avg_val:.4f}  time={time.time()-t0:.1f}s")

    torch.save(midas.state_dict(), os.path.join(CONFIG["checkpoint_dir"], f"midas_epoch{epoch+1}.pt"))
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("\nآموزش تمام شد ✅")


## ۹) نمودار روند Loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history["train_loss"], label="Train Loss", marker="o")
plt.plot(history["val_loss"], label="Validation Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("SSI Loss")
plt.title("روند آموزش")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## ۱۰) ارزیابی نهایی روی داده تست

معیارهای استاندارد تخمین عمق:
- **AbsRel**: میانگین خطای نسبی
- **RMSE**: ریشه میانگین مربعات خطا
- **δ1**: درصد پیکسل‌هایی که نسبت پیش‌بینی به واقعی کمتر از ۱٫۲۵ است (هرچه بیشتر بهتر)

In [ ]:
def evaluate(model, loader):
    model.eval()
    abs_rel_total = rmse_total = delta1_total = 0.0
    n = 0
    with torch.no_grad():
        for img, depth_gt, mask in loader:
            img = img.to(device)
            depth_gt = depth_gt.to(device)
            mask = mask.to(device)

            pred = model(img)
            scale, shift, pred = compute_scale_and_shift(pred, depth_gt, mask)
            pred_aligned = scale.view(-1, 1, 1) * pred + shift.view(-1, 1, 1)

            valid = mask.bool()
            if valid.sum() == 0:
                continue

            p = pred_aligned[valid].clamp(min=1e-3)
            g = depth_gt[valid].clamp(min=1e-3)

            abs_rel_total += torch.mean(torch.abs(p - g) / g).item()
            rmse_total += torch.sqrt(torch.mean((p - g) ** 2)).item()
            ratio = torch.max(p / g, g / p)
            delta1_total += (ratio < 1.25).float().mean().item()
            n += 1

    if n == 0:
        return {"AbsRel": float("nan"), "RMSE": float("nan"), "delta1": float("nan")}
    return {"AbsRel": abs_rel_total / n, "RMSE": rmse_total / n, "delta1": delta1_total / n}

metrics = evaluate(midas, test_loader)
print("نتایج ارزیابی روی داده تست:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")


## ۱۱) نمایش بصری چند پیش‌بینی نمونه

In [ ]:
midas.eval()
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
with torch.no_grad():
    for i in range(min(4, len(test_dataset))):
        img_t, depth_gt, mask = test_dataset[i]
        pred = midas(img_t.unsqueeze(0).to(device))

        scale, shift, pred = compute_scale_and_shift(
            pred,
            depth_gt.unsqueeze(0).to(device),
            mask.unsqueeze(0).to(device)
        )
        pred_aligned = (scale.view(-1, 1, 1) * pred + shift.view(-1, 1, 1)).squeeze().cpu().numpy()

        img_show = img_t.permute(1, 2, 0).numpy()
        img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min() + 1e-6)

        axes[0, i].imshow(img_show)
        axes[0, i].set_title("RGB")
        axes[0, i].axis("off")

        axes[1, i].imshow(depth_gt.numpy(), cmap="inferno")
        axes[1, i].set_title("Depth GT")
        axes[1, i].axis("off")

        axes[2, i].imshow(pred_aligned, cmap="inferno")
        axes[2, i].set_title("Prediction")
        axes[2, i].axis("off")

plt.tight_layout()
plt.show()


## ۱۲) ذخیره مدل نهایی

In [ ]:
final_path = os.path.join(CONFIG["checkpoint_dir"], "midas_finetuned_final.pt")
torch.save(midas.state_dict(), final_path)
print(f"مدل نهایی ذخیره شد در: {final_path}")


## نکات پایانی

- **اگر خطای CUDA Out of Memory گرفتید:** در سلول Config، `batch_size` را به ۱ و `img_size` را به ۲۲۴ کاهش دهید.
- **برای نتیجه بهتر (GPU قوی‌تر یا Colab):** `num_train_samples` و `epochs` را افزایش دهید و `freeze_backbone` را `False` کنید.
- مدل ذخیره‌شده را می‌توانید بعداً این‌گونه بارگذاری کنید:
```python
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas.load_state_dict(torch.load("midas_checkpoints/midas_finetuned_final.pt", map_location="cpu"))
midas.eval()
```

### خلاصه اصلاحات این نسخه نهایی:
1. `revision="refs/convert/parquet"` برای حل خطای Dataset scripts
2. چک وجود داده قبل از دانلود (skip if exists)
3. `num_workers=0` برای حل PicklingError
4. تشخیص امن دستگاه + fallback خودکار به CPU در صورت خطای CUDA kernel
5. هم‌اندازه کردن خودکار خروجی مدل با عمق واقعی داخل loss و evaluate
